### Structured Output

#### Pydantic

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3.8-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.13', 'langchain': '1.4.7'}}, client=<groq.resources.chat.completions.Completions object at 0x00000257C2A59D30>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000257C2A5A7B0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description= "The title of the movie")
    year: int = Field(description= "The year the movie was released")
    director: str = Field(description= "The director of the movie")
    rating: float = Field(description= "The rating of the movie on a scale of 1 to 10")
    

In [5]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.13', 'langchain': '1.4.7'}}, client=<groq.resources.chat.completions.Completions object at 0x00000257C2A59D30>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000257C2A5A7B0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The rating of the movie on a scale of 1 to 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'functi

In [6]:
model.invoke("Please provide information about the movie Inception.")

AIMessage(content='**Inception** is a 2010 science fiction heist film written and directed by **Christopher Nolan**. It is widely regarded as one of the most influential and complex films of the 21st century, known for its intricate plot, mind-bending concepts, and stunning visual effects.\n\n### 🎬 Basic Information\n- **Title**: Inception\n- **Release Date**: July 16, 2010 (USA)\n- **Director**: Christopher Nolan\n- **Main Cast**:\n  - Leonardo DiCaprio (as Dom Cobb)\n  - Joseph Gordon-Levitt (as Arthur)\n  - Ellen Page (as Ariadne)\n  - Tom Hardy (as Eames)\n  - Ken Watanabe (as Saito)\n  - Cillian Murphy (as Fischer)\n  - Marion Cotillard (as Mal Cobb)\n- **Music**: Hans Zimmer (featuring the iconic track *"Time"*)\n- **Runtime**: 148 minutes\n- **Box Office**: Grossed over $833 million worldwide against a $160 million budget.\n\n### 📖 Plot Summary\nIn a near-future world, a technology called **inception** allows skilled thieves to steal secrets from within a person’s subconscious w

In [ ]:
model_with_structure.invoke("Please provide information about the movie Inception.")

##### Message output alongside parsed structure

In [ ]:

from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

##### Nested Structure

In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

#### TypedDict

In [11]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon',
 'file_path': '/tmp/avengers_2012.json',
 'rating': 8,
 'title': 'The Avengers',
 'year': 2012}

In [ ]:

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

In [ ]:
model_withtypedict.profile

In [15]:
model.profile

#### DataClasses

In [16]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='95db28b4-4497-4de8-8e5f-a44675052db6'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'hrn32f4js', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 351, 'total_tokens': 416, 'completion_time': 0.217464095, 'completion_tokens_details': None, 'prompt_time': 0.024201468, 'prompt_tokens_details': None, 'queue_time': 0.031562829, 'total_time': 0.241665563}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_8999620a6e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a168b9-d960-7221-b3f5-9bac6c25cce5-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'john@exa

In [18]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [20]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [21]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')